# **Maternal Health Risk — Notebook 3 of 5: Models (Baselines + Ensembles)**

We train a panel of classifiers with **default / sensible parameters** and compare them on the held-out test set. The best two will be carried into the hyper-tuning notebook.


## **1. Imports & Load Preprocessed Data**

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib, time
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from sklearn.naive_bayes     import GaussianNB
from sklearn.ensemble        import (RandomForestClassifier, GradientBoostingClassifier,
                                     ExtraTreesClassifier, AdaBoostClassifier)
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, classification_report, confusion_matrix,
                                     ConfusionMatrixDisplay)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

RANDOM_STATE = 42
ARTIFACTS = Path('artifacts')
FIG_DIR   = Path('figures'); FIG_DIR.mkdir(exist_ok=True)

bundle = joblib.load(ARTIFACTS/'preprocessed.joblib')
X_train, X_test  = bundle['X_train'], bundle['X_test']
y_train, y_test  = bundle['y_train'], bundle['y_test']
feature_names    = bundle['feature_names']
class_names      = bundle['class_names']
print('Loaded preprocessed bundle. Train', X_train.shape, ' Test', X_test.shape)


## **2. Define the Model Panel**

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree'      : DecisionTreeClassifier(random_state=RANDOM_STATE),
    'k-NN (k=5)'         : KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB'        : GaussianNB(),
    'SVM (RBF)'          : SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    'Extra Trees'        : ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
    'Gradient Boosting'  : GradientBoostingClassifier(random_state=RANDOM_STATE),
    'AdaBoost'           : AdaBoostClassifier(random_state=RANDOM_STATE),
}
if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=5,
        eval_metric='mlogloss', use_label_encoder=False, random_state=RANDOM_STATE)
list(models.keys())


## **3. Train, Predict & Evaluate**

In [ ]:
rows = []
trained = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, average='weighted', zero_division=0),
        'Recall'   : recall_score(y_test, pred, average='weighted', zero_division=0),
        'F1'       : f1_score(y_test, pred, average='weighted', zero_division=0),
        'Time (s)' : round(time.time()-t0, 3),
    })
    trained[name] = model

results = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
results.style.format({'Accuracy':'{:.3f}','Precision':'{:.3f}','Recall':'{:.3f}','F1':'{:.3f}'})


## **4. Visualise Comparison**

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
results_melt = results.melt(id_vars='Model',
                            value_vars=['Accuracy','Precision','Recall','F1'],
                            var_name='Metric', value_name='Score')
sns.barplot(data=results_melt, x='Model', y='Score', hue='Metric', ax=ax)
ax.set_title('Baseline & Ensemble Model Comparison (default params)')
ax.set_ylim(0, 1.05); plt.xticks(rotation=35, ha='right')
plt.tight_layout(); plt.savefig(FIG_DIR/'06_model_comparison.png'); plt.show()


## **5. Detailed Report — Best Model**

In [ ]:
best_name  = results.iloc[0]['Model']
best_model = trained[best_name]
print('Best (untuned) model:', best_name)

y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d'); plt.title(f'Confusion matrix — {best_name}')
plt.tight_layout(); plt.savefig(FIG_DIR/'07_cm_best_untuned.png'); plt.show()


## **6. Save Untuned Results**

In [ ]:
joblib.dump({'results': results, 'best_name': best_name,
             'best_model': best_model}, ARTIFACTS/'models_untuned.joblib')
print('Saved:', ARTIFACTS/'models_untuned.joblib')


**Takeaway.** Tree-based ensembles (Random Forest, Extra Trees, XGBoost) typically dominate this dataset. We carry the top two into the hyper-tuning notebook.
